# Initialization

In [ ]:
# import block
import csv, glob, os, pandas, serial, serial.tools.list_ports, shutil, time
from datetime import datetime

# Functions

## Data Processing

In [ ]:
def portInspector(printInfo = True):
    # query ports
    ports = serial.tools.list_ports.comports()
    portDictionary = {}

    # set up report on them
    for port in ports:
        portDictionary[port.device] = port.description
        if printInfo == True:
            print(f"Device: {port.device} | Description: {port.description}")

    # return the keys for the dropdown widget
    return sorted(list(portDictionary.keys()))

In [ ]:
# one-time trial runner

def oneTrialRunner(expectedPort,rateBAUD, expectedSensors, trialsDirectory):
    """
    """
    # universal vars
    rawPayload = []
    outputPayload = []
    headerFound = False

    # check that the user actually selected a port
    currentPorts = portInspector(printInfo = False)
    if expectedPort not in currentPorts:
        raise ValueError("Either you didn't run the port code (above), or you didn't select a port.")

    # open serial line
    ser = serial.Serial(expectedPort, rateBAUD, timeout = 1)
    time.sleep(2)
    ser.reset_input_buffer()
    print(f"Connected to {expectedPort}. Press the 'Stop' button in Jupyter to halt logging.\n")

    # check the payload
    while True:
        # read line by line
        rawBytes = ser.readline()
        if not rawBytes:
            continue
        line = rawBytes.decode('utf-8').strip()
        # pull matching payload if the last loop found the header
        if headerFound == True:
            lineParts = line.split(',')
            for value in lineParts:
                if value.isdigit() == True:
                    rawPayload.append(int(value))
            break # break out of loop

        # check if this line is header
        receivedPins = []
        lineParts = line.split(',')
        for item in lineParts:
            if item.isdigit():
                receivedPins.append(int(item))
        if receivedPins == expectedSensors:
            headerFound = True
        # raise value error in the event that the user didn't bother reading the documentation and is just changing things
        elif len(receivedPins) > 0 and len(receivedPins) != len(expectedSensors):
            ser.close() # release port for next iteration
            raise ValueError(
                f"Sensors received: {line}; sensors expected: {expectedSensors}.\n"
                f"If you want to change the number of sensors, you need to fix the arduino sketch and send it to the microcontroller, first.\n"
                f"Check the documentation.\n"
                )

    # fix timestamps for compilation and calculation in later steps
    for millisecondValue in rawPayload:
        seconds = millisecondValue / 1000.0
        outputPayload.append(seconds)

    # to file
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    fileName = f"{trialsDirectory}/trial-{timestamp}.csv"
    with open(fileName, "w", newline = "") as file:
        writer = csv.writer(file)
        writer.writerow(expectedSensors)
        writer.writerow(outputPayload)

    # return the payload for the user to see
    ser.close() # release port for next iteration
    return outputPayload

In [ ]:
# follow-mode trial runner
def multiTrialRunner(expectedPort, rateBAUD, expectedSensors, trialsDirectory, maxTrials = None):
    """
    """
    # universal vars
    allTrialsPayloads = []
    trialCount = 0
    # port check is already in oneTrialRunner
    print("Logging multiple trials. Press the stop button (square) to the left in Jupyter to exit.")

    # loop through the one trial runner
    try:
        # endless loop til user stops the trials
        while True:
            if maxTrials != None:
                if trialCount >= maxTrials:
                    print(f"Completed requested trials: {maxTrials}.")
                    break
            # increase the counter with each iteration
            trialCount += 1
            print(f"\nRunning trial: {trialCount}")
            # run the trial
            tempPayload = oneTrialRunner(expectedPort, rateBAUD, expectedSensors, directoryForOutput)
            allTrialsPayloads.append(tempPayload)

    # catch the stop
    except KeyboardInterrupt:
        print(f"\n[Stopped] - {len(allTrialsPayloads)} total trial(s) were run.")

    return allTrialsPayloads


In [ ]:
def aggregateTrialsGetSpeed(trialsDirectory, directoryForAggregate):
    """
    """
    outputDataframes = []
    rawFiles = glob.glob(f"{trialsDirectory}/*")
    rawDataframes = []
    # get pandas dataframes per file
    for file in rawFiles:
        tempDF = pandas.read_csv(file, delimiter = ',', index_col = False)
        rawDataframes.append(tempDF)
    # use the headers as rows, instead, and iterate through with each file representing a trial
    # for the length of the trials

    # then, calculate speed per row; this'll be output dataframe one

    # then, calculate average speed, change in speed over time, per trial

    # option for just raw dataframe output (trial # headers after pins with times)



    # output them all to files
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    fileName = f"{directoryForOutput}/trial-{timestamp}.csv"
    with open(fileName, "w", newline = "") as file:
        writer = csv.writer(file)
        writer.writerow(expectedSensors)
        writer.writerow(outputPayload)

    # return all the dataframes as a list of dataframes for the vis software below
    return outputDataframes


## Data Visualization